# MODE 2 — M2_F01 ANIMATION VALIDATOR — PRODUCTION

> Phase 8 — Dual Pipeline Doctrine — v1.0.0

**Notebook de production unique — une cellule par étape.**

```
INPUT  : IN_GLB_AVATAR/avatar.glb + IN_AUDIO/audio.wav (optionnel)
OUTPUT : OUT_VALIDATED/avatar_validated.glb + OUT_REPORT/m2_f01_report.json
```

In [ ]:
#@title 🔗 [EXODUS] Drive + Session JSON
#@markdown Monte le Drive et lit exodus_session.json genere par EXO_LAUNCHER
from google.colab import drive
drive.mount('/content/drive')

import sys, json
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/EXODUS_V2"  #@param {type:"string"}
sys.path.insert(0, DRIVE_ROOT)

_session_path = Path(DRIVE_ROOT) / "exodus_session.json"
if _session_path.exists():
    with open(_session_path) as _f:
        EXODUS_SESSION = json.load(_f)
    print("OK exodus_session.json charge")
    print(f"  Mode     : {EXODUS_SESSION['mode']} --- {EXODUS_SESSION['mode_label']}")
    print(f"  Timestamp: {EXODUS_SESSION['timestamp']}")
    print(f"  Drive    : {EXODUS_SESSION['drive_root']}")
else:
    print("ATTENTION : exodus_session.json introuvable")
    print("   -> Lancer EXO_LAUNCHER.ipynb d'abord.")
    EXODUS_SESSION = {
        "mode": None, "mode_label": "UNKNOWN",
        "drive_root": DRIVE_ROOT, "status": "missing"
    }

In [ ]:
# ── CELLULE 0 — CONFIGURATION OPÉRATEUR ──────────────────────
# Modifier ces valeurs si nécessaire

GLB_FILE   = None   # None = auto-détection IN_GLB_AVATAR/*.glb
AUDIO_FILE = None   # None = auto-détection IN_AUDIO/* (optionnel)
DRY_RUN    = False  # True = valide sans copier
VERBOSE    = True

In [ ]:
# ── CELLULE 1 — LANCEMENT VALIDATION ─────────────────────────
import subprocess, sys
from pathlib import Path

script = Path("EXO_M2_F01_ANIMATION.py")
cmd = [sys.executable, str(script)]

if GLB_FILE:
    cmd += ["--glb", GLB_FILE]
if AUDIO_FILE:
    cmd += ["--audio", AUDIO_FILE]
if DRY_RUN:
    cmd.append("--dry-run")
if VERBOSE:
    cmd.append("--verbose")

print(f"Commande : {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=False, text=True)
print(f"\nCode retour : {result.returncode}")

In [ ]:
# ── CELLULE 2 — LECTURE RAPPORT ───────────────────────────────
import json
from pathlib import Path

report_path = Path("../OUT_REPORT/m2_f01_report.json")
if not report_path.exists():
    print("Rapport introuvable — vérifier la cellule 1")
else:
    with open(report_path) as f:
        r = json.load(f)

    status_icon = "✅" if r["status"] == "SUCCESS" else "❌"
    print(f"{status_icon} STATUS : {r['status']}")
    print(f"   Timestamp    : {r['timestamp']}")

    glb = r.get("glb_validation", {})
    print(f"   GLB valide   : {glb.get('valid')}")
    print(f"   Taille       : {glb.get('file_size_mb')} MB")
    print(f"   Animations   : {len(glb.get('animations', []))}")
    for anim in glb.get("animations", []):
        print(f"     - {anim['name']} : {anim['duration_s']:.3f}s")
    print(f"   Durée totale : {glb.get('total_duration_s', 0):.3f}s")

    r03 = r.get("loi_r03", {})
    r03_icon = "✅" if r03.get("status") in ("CONFORME", "NOT_APPLICABLE", "SKIPPED") else "❌"
    print(f"\n   {r03_icon} LOI R-03 : {r03.get('status')}")
    print(f"   {r03.get('message', '')}")

    if r.get("errors"):
        print(f"\n   ERREURS : {r['errors']}")
    if r.get("warnings"):
        print(f"   AVERTISSEMENTS : {r['warnings']}")

    if r["status"] == "SUCCESS":
        print("\n   ──────────────────────────────────")
        print("   ✅ Prêt pour M2_F02 ─► OUT_VALIDATED/")
        print("   Transférer manuellement OUT_VALIDATED/ vers 08_M2_F02_LOGISTICS/IN_*/")